*Built with Grok Build*

# Homework Module 2 — Estimating Quad Accuracy (Finite Integral)

Course: **EEE591 / EEE419 Python for Rapid Engineering Solutions**.

Executable walkthrough of `HW2_Estimating_Quad_Accuracy.py` (Problem 1).

We compare **SciPy `quad`** against a **hand-written composite trapezoidal rule** for
$$
I_1 = \int_{-4}^{5} \bigl(a x^5 + b x^2 - c\bigr)\,dx
$$
for given coefficients $a$, $b$, and $c$. The original script used `input()`; here we hard-code
a representative triple so the notebook runs non-interactively.


## Imports and problem coefficients

Use a fixed example triple in $[-10, 10]$ (the same style of input the assignment expects).


In [1]:
import numpy as np
from scipy.integrate import quad

# Original script: input_str = input("Input a set of 3 numbers between -10 and 10: ")
input_str = "-2 6 0"  # non-interactive demo values (a, b, c)
print("Input coefficients string:", repr(input_str))


Input coefficients string: '-2 6 0'


## Parse coefficients $a$, $b$, $c$

Split on spaces, strip whitespace, and cast to `float` — matching the CLI parsing in the graded script.


In [2]:
split_stripped_str_list_1 = input_str.split(" ")
split_stripped_str_list_2 = [item.strip() for item in split_stripped_str_list_1]

a = float(split_stripped_str_list_2[0])
b = float(split_stripped_str_list_2[1])
c = float(split_stripped_str_list_2[2])

print(f"a = {a}, b = {b}, c = {c}")


a = -2.0, b = 6.0, c = 0.0


## Integration setup

Finite interval $[-4, 5]$ and one million trapezoidal subintervals (as in the assignment script).


In [3]:
NUM_POINTS = 1_000_000
LOWER_LIMIT = -4
UPPER_LIMIT = 5

def integrand(x):
    """I1(x) = a x^5 + b x^2 - c."""
    return a * x**5 + b * x**2 - c

print(f"Limits: [{LOWER_LIMIT}, {UPPER_LIMIT}], trapezoid points: {NUM_POINTS:,}")


Limits: [-4, 5], trapezoid points: 1,000,000


## Method 1 — SciPy `quad`

`quad` is an adaptive quadrature routine (QUADPACK). We treat it as the more accurate reference method.


In [4]:
integral_value_quad, integral_error_quad = quad(integrand, LOWER_LIMIT, UPPER_LIMIT)
print(f"Method 1 (quad): I1 = {integral_value_quad: .4f}")
print(f"quad absolute error estimate: {integral_error_quad:.3e}")


Method 1 (quad): I1 = -3465.0000
quad absolute error estimate: 7.171e-11


## Method 2 — composite trapezoidal rule

With $N$ subintervals of width $h = (b-a)/N$,
$$
\int_a^b f(x)\,dx \approx \frac{h}{2}\Bigl(f(a) + 2\sum_{i=1}^{N-1} f(a+ih) + f(b)\Bigr).
$$
This is implemented explicitly (no NumPy vectorization) to mirror the original homework code.


In [5]:
subinterval_width = (UPPER_LIMIT - LOWER_LIMIT) / NUM_POINTS

# Trapezoidal rule: (h/2) * (f(a) + f(b) + 2 * sum interior)
initial_result = 0.5 * (integrand(LOWER_LIMIT) + integrand(UPPER_LIMIT))
running_sum = initial_result

for i in range(1, NUM_POINTS):
    x = LOWER_LIMIT + i * subinterval_width
    f_x = integrand(x)
    running_sum += 2 * f_x

final_result_numerical = running_sum * (subinterval_width / 2)
print(f"Method 2 (trapezoidal): I1 = {final_result_numerical: .4f}")
print(f"Subinterval width h = {subinterval_width:.6e}")


Method 2 (trapezoidal): I1 = -3464.9911
Subinterval width h = 9.000000e-06


## Percentage error between methods

$$
\text{percentage error} = \frac{|I_{\text{quad}} - I_{\text{trap}}|}{|I_{\text{quad}}|} \times 100.
$$


In [6]:
error_percentage = (abs(integral_value_quad - final_result_numerical) / abs(integral_value_quad)) * 100
print(f"Method 1: I1 = {integral_value_quad: .4f}")
print(f"Method 2: I1 = {final_result_numerical: .4f}")
print(f"Percentage error: {error_percentage: .4f}%")


Method 1: I1 = -3465.0000
Method 2: I1 = -3464.9911
Percentage error:  0.0003%


## Takeaways

- Adaptive `quad` is usually far more accurate *and* cheaper than a fixed-grid trap rule with $10^6$ panels.
- Report both values and the relative error so graders can see numerical agreement.
- Original CLI script: `HW2_Estimating_Quad_Accuracy.py` (and the first half of `hw2.py`).
